# Food Image Data Processing Pipeline

This notebook processes raw food images for training a classification model. The pipeline performs:

1. **Quality filtering** - Removes tiny, blurry, or oddly-shaped images
2. **Deduplication** - Removes near-duplicate images using perceptual hashing
3. **Train/Val/Test splitting** - Stratified 80/10/10 split
4. **Data organization** - Creates clean directory structure with manifest

## Configuration & Imports

Set up libraries and define processing parameters:
- **Input/Output paths**: Read from `smaller_dataset/`, write to `data_clean/`
- **Quality thresholds**: Minimum size (224px), max aspect ratio (2.0), blur threshold (80.0)
- **Deduplication**: Perceptual hash distance ≤ 5 considered duplicates
- **Split ratios**: 80% train, 10% validation, 10% test

In [1]:
# --- Libraries ---
import os, sys, csv, math, random, shutil, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple
from PIL import Image, ImageOps, UnidentifiedImageError
import imagehash, cv2, pandas as pd
from tqdm.notebook import tqdm

# --- Configuration ---
DATA_DIR = Path("smaller_dataset")       
OUT_DIR  = Path("data_clean")   

MIN_SIDE       = 224              
MAX_AR         = 2.0             
BLUR_THRESHOLD = 80.0         
PHASH_DIST_MAX = 5             
SPLIT_RATIOS   = {"train":0.8,"val":0.1,"test":0.1}
RANDOM_SEED    = 42
LINK_MODE      = "symlink"       

# Optional label mapping (synonyms → canonical)
LABEL_MAP: Dict[str,str] = {
    
}

## Data Classes & Helper Functions

Defines the `Sample` dataclass to store metadata for each image:
- File path, class label, source folder
- Image dimensions, perceptual hash, blur variance
- Keep/drop status with reason, and assigned split

Helper functions handle:
- **Label canonicalization** - Maps synonyms to standard class names
- **Class/source inference** - Extracts class from folder structure
- **Aspect ratio validation** - Checks if image proportions are acceptable
- **Hamming distance** - Compares perceptual hashes for deduplication

In [2]:
@dataclass
class Sample:
    path: str
    cls: str
    source: str
    width: int
    height: int
    phash_hex: str
    blur_var: float
    kept: bool
    drop_reason: str
    split: str

def canonical_label(name:str)->str:
    return LABEL_MAP.get(name.strip(), name.strip())

def infer_class_and_source(p:Path)->Tuple[str,str]:
    parts = p.relative_to(DATA_DIR).parts
    if len(parts)>=3:
        src, cls = parts[0], parts[1]
    elif len(parts)>=2:
        src, cls = "unknown", parts[0]
    else:
        src, cls = "unknown", "unknown"
    return canonical_label(cls), src

def aspect_ratio_ok(w:int,h:int,max_ar:float)->bool:
    if w==0 or h==0: return False
    ar = max(w,h)/max(1,min(w,h))
    return ar<=max_ar

def hamming_dist_hex(a:str,b:str)->int:
    if not a or not b: return 999
    return (int(a,16)^int(b,16)).bit_count()

## Image Analysis Functions

- **`read_image_info()`** - Opens each image and extracts:
  - Width & height (with EXIF rotation correction)
  - Perceptual hash (pHash) for duplicate detection
  - Blur variance using Laplacian operator (higher = sharper)

- **`collect_images()`** - Recursively finds all image files (jpg, png, bmp, webp)

In [3]:
def read_image_info(img_path:Path)->Tuple[int,int,str,float]:
    try:
        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im)
            w,h = im.size
            ph = imagehash.phash(im)
            ph_hex = str(ph)
    except UnidentifiedImageError:
        return 0,0,"",0.0
    except Exception:
        return 0,0,"",0.0

    img_cv = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    blur = cv2.Laplacian(img_cv, cv2.CV_64F).var() if img_cv is not None else 0.0
    return w,h,ph_hex,float(blur)

def collect_images(root:Path):
    exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    return [p for p in root.rglob("*") if p.suffix.lower() in exts and p.is_file()]

## Core Processing Pipeline

### `initial_scan()`
Scans all images and filters based on quality criteria:
- **Tiny images** - Smaller than 224px on any side
- **Extreme aspect ratio** - Width/height ratio > 2.0
- **Blurry images** - Laplacian variance < 80

### `deduplicate()`
Removes near-duplicate images by comparing perceptual hashes:
- Images with hamming distance <= 5 are considered duplicates
- Keeps the first occurrence, drops later duplicates

### `split_by_source()`
Stratified train/val/test splitting:
- Groups images by class and source folder
- Ensures images from same source stay in same split (prevents data leakage)
- Falls back to random split if source is unknown

In [4]:
def initial_scan()->List[Sample]:
    samples=[]
    for p in tqdm(collect_images(DATA_DIR), desc="Scanning images"):
        cls,src = infer_class_and_source(p)
        w,h,ph,blur = read_image_info(p)
        keep,reason = True,""
        if min(w,h)<MIN_SIDE: keep,reason=False,f"tiny<{MIN_SIDE}"
        elif not aspect_ratio_ok(w,h,MAX_AR): keep,reason=False,"extreme_AR"
        elif blur<BLUR_THRESHOLD: keep,reason=False,f"blurry<{BLUR_THRESHOLD}"
        samples.append(Sample(str(p),cls,src,w,h,ph,blur,keep,reason,""))
    return samples

def deduplicate(samples:List[Sample]):
    kept_hashes=[]
    for s in tqdm(samples,desc="De-duping"):
        if not s.kept or not s.phash_hex: continue
        dup=False
        for h in kept_hashes:
            if hamming_dist_hex(s.phash_hex,h)<=PHASH_DIST_MAX:
                dup=True;break
        if dup:
            s.kept=False; s.drop_reason=f"near-dup≤{PHASH_DIST_MAX}"
        else:
            kept_hashes.append(s.phash_hex)

def split_by_source(samples:List[Sample]):
    random.seed(RANDOM_SEED)
    groups={}
    for i,s in enumerate(samples):
        if not s.kept: continue
        groups.setdefault((s.cls,s.source),[]).append(i)
    class_srcs={}
    for (cls,src) in groups.keys():
        class_srcs.setdefault(cls,[]).append(src)

    for cls,srcs in class_srcs.items():
        uniq=list(set(srcs))
        if uniq==["unknown"]:
            idxs=[i for (c,s),v in groups.items() if c==cls for i in v]
            random.shuffle(idxs)
            n=len(idxs); ntr=int(n*SPLIT_RATIOS["train"]); nval=int(n*SPLIT_RATIOS["val"])
            for j,k in enumerate(idxs):
                samples[k].split="train" if j<ntr else "val" if j<ntr+nval else "test"
        else:
            rs=uniq[:]; random.shuffle(rs)
            nt=max(1,int(len(rs)*SPLIT_RATIOS["train"]))
            nv=max(0,int(len(rs)*SPLIT_RATIOS["val"]))
            train=set(rs[:nt]); val=set(rs[nt:nt+nv]); test=set(rs[nt+nv:])
            for (c,s),idxs in groups.items():
                if c!=cls: continue
                sp="train" if s in train else "val" if s in val else "test"
                for k in idxs: samples[k].split=sp

## Output Generation

Creates the clean dataset structure:
1. **Manifest CSV** - Complete record of all samples with metadata and keep/drop status
2. **Directory structure** - Organizes kept images into `data_clean/train|val|test/<class>/`
3. **Symlinks or copies** - Uses symlinks by default (saves disk space), falls back to copying

In [5]:
def write_manifest_and_materialize(samples:List[Sample]):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = OUT_DIR/"manifest.csv"
    pd.DataFrame([asdict(s) for s in samples]).to_csv(manifest,index=False)

    for s in samples:
        if not s.kept or not s.split: continue
        src=Path(s.path); dst_dir=OUT_DIR/s.split/s.cls
        dst_dir.mkdir(parents=True,exist_ok=True)
        dst=dst_dir/src.name
        if LINK_MODE=="copy":
            if not dst.exists(): shutil.copy2(src,dst)
        else:
            if not dst.exists():
                try: os.symlink(os.path.abspath(src),dst)
                except OSError: shutil.copy2(src,dst)
    print(f"✅ Manifest written to {manifest}")
    print(f"✅ Clean data at {OUT_DIR}/train|val|test/<class>/")

## Run the Pipeline

Execute all 4 steps:
1. **Scan** - Analyze all images and apply quality filters
2. **Deduplicate** - Remove near-duplicate images
3. **Split** - Assign train/val/test splits
4. **Write** - Generate manifest and organize files

In [6]:
print("[1/4] Scanning ...")
samples = initial_scan()

print("[2/4] De-duplicating ...")
deduplicate(samples)

print("[3/4] Splitting ...")
split_by_source(samples)

print("[4/4] Writing output ...")
write_manifest_and_materialize(samples)

[1/4] Scanning ...


Scanning images:   0%|          | 0/9211 [00:00<?, ?it/s]

c:\schoolwork\computer_vis\.venv\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[2/4] De-duplicating ...


De-duping:   0%|          | 0/9211 [00:00<?, ?it/s]

[3/4] Splitting ...
[4/4] Writing output ...
✅ Manifest written to data_clean\manifest.csv
✅ Clean data at data_clean/train|val|test/<class>/
✅ Manifest written to data_clean\manifest.csv
✅ Clean data at data_clean/train|val|test/<class>/


## Results Analysis

Review the processing results:
- **Total samples** - How many images were processed
- **Kept vs dropped** - How many passed quality filters
- **Drop reasons** - Why images were rejected (tiny, blurry, duplicate, etc.)
- **Split distribution** - Number of images per class in train/val/test

In [7]:
# Load manifest
df = pd.read_csv(OUT_DIR/"manifest.csv")
print("Rows:", len(df))
display(df.head())

# Check kept vs dropped
display(df['kept'].value_counts())
print("\nDrop reasons:")
display(df.loc[~df['kept'],'drop_reason'].value_counts())

# Split distribution
print("\nSplit distribution per class:")
display(df[df['kept']].groupby(['split','cls']).size().unstack(fill_value=0))

Rows: 9211


,path,cls,source,width,height,phash_hex,blur_var,kept,drop_reason,split
0,smaller_dataset\Braised Crucian Carp\1.jpg,Braised Crucian Carp,unknown,800,1076,edec90cd85d20f92,1393.952019,True,NaN,train
1,smaller_dataset\Braised Crucian Carp\10.jpg,Braised Crucian Carp,unknown,667,500,fbd1b80ad16a942b,564.732656,True,NaN,train
2,smaller_dataset\Braised Crucian Carp\100.jpg,Braised Crucian Carp,unknown,450,800,f288ed57e87a0961,1166.641026,True,NaN,train
3,smaller_dataset\Braised Crucian Carp\101.jpg,Braised Crucian Carp,unknown,640,427,f854a94a256f61d6,1560.950554,True,NaN,test
4,smaller_dataset\Braised Crucian Carp\102.jpg,Braised Crucian Carp,unknown,380,380,d57e54e86e85809e,1593.365864,True,NaN,train


kept
True     8238
False     973
Name: count, dtype: int64


Drop reasons:


drop_reason
blurry<80.0    382
near-dup≤5     378
tiny<224       128
extreme_AR      85
Name: count, dtype: int64


Split distribution per class:


cls,Braised Crucian Carp,Braised Pork,Dry Pot Cauliflower,Egg Drop Soup,Fish Flavored Shredded Pork,Fish Head Tofu Soup,Fragrant Soy-sauced Eggplant,Fried Milk Custard,Glass Noodles with Minced Pork,Hot and Sour Lotus Root Slices,Kung Pao Chicken,Scallion Oil Noodles,Spicy Chongqing Hotpot,Spicy Crab,Spicy Tofu,Steamed Pork with Rice Flour,Stir-fried Pork Kidney,Stir-fried Shredded Potatoes,Tomato Scrambled Eggs,Winter Melon Pork Rib Soup
split,,,,,,,,,,,,,,,,,,,,
test,29,97,29,26,79,28,28,24,29,27,80,30,30,28,28,28,27,82,83,28
train,224,770,226,199,625,219,217,188,224,209,633,231,230,215,216,224,211,645,664,212
val,28,96,28,24,78,27,27,23,28,26,79,28,28,26,27,28,26,80,83,26


## Data Processing Insights

### Overall Statistics
| Metric | Value |
|--------|-------|
| **Total images scanned** | 9,211 |
| **Images kept** | 8,238 (89.4%) |
| **Images dropped** | 973 (10.6%) |

### Drop Reasons Breakdown
| Reason | Count | Description |
|--------|-------|-------------|
| Blurry | 382 | Laplacian variance < 80 |
| Near-duplicates | 378 | Perceptual hash distance <= 5 |
| Tiny | 128 | Smaller than 224px |
| Extreme aspect ratio | 85 | Width/height ratio > 2.0 |

### Class Distribution (Training Set)
- **Largest classes**: Braised Pork (770), Tomato Scrambled Eggs (664), Stir-fried Shredded Potatoes (645)
- **Smallest classes**: Fried Milk Custard (188), Egg Drop Soup (199)
- **Average per class**: ~330 images

### Key Observations
1. **Drop rate is healthy** - 10.6% dropped is reasonable for quality filtering
2. **Dataset size is sufficient** - ~6,600 training images across 20 classes works well with transfer learning
3. **Class imbalance exists** - ~4x difference between largest and smallest classes
4. **Test/Val sets adequate** - 24-97 images per class for evaluation